**02_eda.ipynb**
+ bureau.csv analizi
+ previous_application.csv analizi
+ join kontrolleri
+ feature store kontrolleri


In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(project_root)

c:\Users\ASUS\creditguard-ai


In [3]:
import pandas as pd

bureau = pd.read_csv(
    "../data/raw/bureau.csv"
)

print(bureau.shape)
bureau.head()

(1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [4]:
bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           object 
 3   CREDIT_CURRENCY         object 
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             object 
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), object(3)
memory usage: 222.6+ MB


In [5]:
bureau[
    [
        "SK_ID_CURR",
        "SK_ID_BUREAU",
        "CREDIT_ACTIVE",
        "AMT_CREDIT_SUM",
        "AMT_CREDIT_SUM_DEBT"
    ]
].head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT
0,215354,5714462,Closed,91323.0,0.0
1,215354,5714463,Active,225000.0,171342.0
2,215354,5714464,Active,464323.5,NaN
3,215354,5714465,Active,90000.0,NaN
4,215354,5714466,Active,2700000.0,NaN


In [10]:


bureau = pd.read_csv(
    "../data/raw/bureau.csv"
)

from src.features.build_bureau_features import (
    create_bureau_features
)

bureau_features = create_bureau_features(
    bureau
)

bureau_features.head()

,SK_ID_CURR,bureau_total_credit,bureau_total_debt,bureau_active_loans,bureau_closed_loans,bureau_overdue_amount,bureau_debt_credit_ratio
0,100001,1453365.000,596686.5,3,4,0.0,0.410555
1,100002,865055.565,245781.0,2,6,0.0,0.284122
2,100003,1017400.500,0.0,1,3,0.0,0.000000
3,100004,189037.800,0.0,0,2,0.0,0.000000
4,100005,657126.000,568408.5,2,1,0.0,0.864992


In [11]:
bureau_features.shape

(305811, 7)

In [12]:
train = pd.read_csv(
    "../data/raw/application_train.csv"
)

print(train.shape)

(307511, 122)


In [15]:
train_bureau = train.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left"
)

print(train_bureau.shape)
# 122 orijinal kolon + 6 yeni bureau feature = 128 kolon

(307511, 128)


In [16]:
# Muhtemelen bazı müşterilerin bureau kaydı olmadığı için eksikler göreceğiz.
train_bureau[
    [
        "bureau_total_credit",
        "bureau_total_debt",
        "bureau_active_loans",
        "bureau_closed_loans"
    ]
].isnull().mean()

bureau_total_credit    0.143149
bureau_total_debt      0.143149
bureau_active_loans    0.143149
bureau_closed_loans    0.143149
dtype: float64

**yaklaşık:** %14.3 müşterinin bureau geçmişi yok.

Anlamı: Kredi bürosunda kayıtlı geçmiş kredi bilgisi bulunmuyor.


İleride eklenecek feature: **has_bureau_history**

In [17]:
train_bureau[
    [
        "bureau_total_credit",
        "bureau_total_debt",
        "bureau_debt_credit_ratio"
    ]
].describe()

c:\Users\ASUS\creditguard-ai\venv\Lib\site-packages\numpy\core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,bureau_total_credit,bureau_total_debt,bureau_debt_credit_ratio
count,2.634910e+05,2.634910e+05,2.624720e+05
mean,1.955807e+06,6.406503e+05,NaN
std,4.101728e+06,1.633961e+06,NaN
min,0.000000e+00,-6.981558e+06,-inf
25%,3.433773e+05,0.000000e+00,0.000000e+00
50%,9.617040e+05,1.690200e+05,2.091433e-01
75%,2.297721e+06,6.600621e+05,4.830696e-01
max,1.017958e+09,3.344983e+08,inf


In [ ]:
previous = pd.read_csv(
    "../data/raw/previous_application.csv" 
      #"Bu müşteri geçmişte Home Credit'e kaç kez başvurdu?"
)

print(previous.shape)

(1670214, 37)


In [19]:
previous.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  object 
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  object 
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  object 
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-nu

In [20]:
from src.features.build_previous_features import (
    create_previous_features
)

previous_features = create_previous_features(
    previous
)

previous_features.shape

(338857, 9)

**application_train :** 307.511 müşteri

**bureau            :** 305.811 müşteri

**previous_app      :** 338.857 müşteri

In [21]:
previous_features.head()

,SK_ID_CURR,prev_application_count,prev_approved_count,prev_refused_count,prev_canceled_count,prev_avg_application_amount,prev_avg_credit_amount,approval_rate,refusal_rate
0,100001,1,1,0,0,24835.50,23787.00,1.0,0.0
1,100002,1,1,0,0,179055.00,179055.00,1.0,0.0
2,100003,3,3,0,0,435436.50,484191.00,1.0,0.0
3,100004,1,1,0,0,24282.00,20106.00,1.0,0.0
4,100005,2,1,0,1,22308.75,20076.75,0.5,0.0


In [22]:
previous_features[
    [
        "prev_application_count",
        "approval_rate",
        "refusal_rate"
    ]
].describe()

,prev_application_count,approval_rate,refusal_rate
count,338857.000000,338857.000000,338857.00000
mean,4.928964,0.744495,0.11142
std,4.220716,0.263250,0.18373
min,1.000000,0.000000,0.00000
25%,2.000000,0.500000,0.00000
50%,4.000000,0.777778,0.00000
75%,7.000000,1.000000,0.20000
max,77.000000,1.000000,1.00000


### 📊 Başvuru Sayısı İstatistikleri (`prev_application_count`)

* **Ortalama (Mean):** 4.93
* **Medyan (Median):** 4
* **Maksimum (Max):** 77

> **💡 Özet Bulgular:**
> * **Ortalama Müşteri:** Yaklaşık **5 kez** başvurmuş.
> * **Uç Değerler:** Bazı müşterilerde bu sayı **77 başvuruya** kadar çıkmaktadır.

---

### 📈 Başvuru Durum Oranları

* **Onaylanma Oranı (Approval Rate):** `Mean = 0.744` (Başvuruların **%74.4'ü** onaylanmış)
* **Reddedilme Oranı (Refusal Rate):** `Mean = 0.111` (Ortalama reddedilme oranı **%11.1**)


In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/train_feature_store.parquet"
)

print(df.shape)

(307511, 150)


In [2]:
df.columns[-20:].tolist()

['credit_per_child',
 'income_credit_difference',
 'annuity_credit_ratio',
 'is_car_owner',
 'is_realty_owner',
 'is_employed',
 'bureau_total_credit',
 'bureau_total_debt',
 'bureau_active_loans',
 'bureau_closed_loans',
 'bureau_overdue_amount',
 'bureau_debt_credit_ratio',
 'prev_application_count',
 'prev_approved_count',
 'prev_refused_count',
 'prev_canceled_count',
 'prev_avg_application_amount',
 'prev_avg_credit_amount',
 'approval_rate',
 'refusal_rate']

**Bureau Feature'ları**

+ bureau_total_credit
+ bureau_total_debt
+ bureau_active_loans
+ bureau_closed_loans
+ bureau_overdue_amount
+ bureau_debt_credit_ratio

> Model, "Bu müşterinin başka bankalardaki borç durumu nedir?" sorusunu görebilecek.

**Previous Application Feature'ları**
+ prev_application_count
+ prev_approved_count
+ prev_refused_count
+ approval_rate
+ refusal_rate

> Model , ""Bu müşteri geçmişte kredi başvurularında nasıl davranmış?" bilgisini görebilecek."

In [3]:
df.select_dtypes(include="object").columns.tolist()

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

In [6]:
import numpy as np
df = pd.read_parquet(
    "../data/processed/train_feature_store.parquet"
)

print(
    np.isinf(
        df.select_dtypes(include=np.number)
    ).sum().sum()
)

NameError: name 'pd' is not defined

In [2]:
import pandas as pd

installments = pd.read_csv(
    "../data/raw/installments_payments.csv"
)

installments.shape
installments.head()
installments.info()
installments.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB


,count,mean,std,min,25%,50%,75%,max
SK_ID_PREV,13605401.0,1.903365e+06,536202.905546,1000001.0,1434191.000,1896520.000,2369094.000,2843499.000
SK_ID_CURR,13605401.0,2.784449e+05,102718.310411,100001.0,189639.000,278685.000,367530.000,456255.000
NUM_INSTALMENT_VERSION,13605401.0,8.566373e-01,1.035216,0.0,0.000,1.000,1.000,178.000
NUM_INSTALMENT_NUMBER,13605401.0,1.887090e+01,26.664067,1.0,4.000,8.000,19.000,277.000
DAYS_INSTALMENT,13605401.0,-1.042270e+03,800.946284,-2922.0,-1654.000,-818.000,-361.000,-1.000
DAYS_ENTRY_PAYMENT,13602496.0,-1.051114e+03,800.585883,-4921.0,-1662.000,-827.000,-370.000,-1.000
AMT_INSTALMENT,13605401.0,1.705091e+04,50570.254429,0.0,4226.085,8884.080,16710.210,3771487.845
AMT_PAYMENT,13602496.0,1.723822e+04,54735.783981,0.0,3398.265,8125.515,16108.425,3771487.845


In [3]:
from src.features.build_installment_features import (
    build_installment_features
)

In [4]:
installment_features = (
    build_installment_features(
        installments
    )
)

In [5]:
installment_features.shape

(339587, 8)

**previous_features** → 338857

**installment_features** → 339587

In [6]:
installment_features.head()

,SK_ID_CURR,installment_count,avg_days_late,max_days_late,late_payment_count,late_payment_ratio,avg_payment_ratio,total_payment_amount
0,100001,7,1.571429,11.0,1,0.142857,1.0,41195.925
1,100002,19,0.000000,0.0,0,0.000000,1.0,219625.695
2,100003,25,0.000000,0.0,0,0.000000,1.0,1618864.650
3,100004,3,0.000000,0.0,0,0.000000,1.0,21288.465
4,100005,9,0.111111,1.0,1,0.111111,1.0,56161.845


In [7]:
installment_features.describe().T

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,339587.0,278154.892278,102880.492598,100001.000000,189042.500000,278238.000000,367315.500000,4.562550e+05
installment_count,339587.0,40.064552,41.053343,1.000000,12.000000,25.000000,51.000000,3.720000e+02
avg_days_late,339578.0,1.027695,8.821114,0.000000,0.000000,0.035714,0.500000,1.885386e+03
max_days_late,339578.0,17.844109,108.275247,0.000000,0.000000,1.000000,9.000000,2.884000e+03
late_payment_count,339587.0,3.376658,6.374030,0.000000,0.000000,1.000000,4.000000,1.590000e+02
late_payment_ratio,339587.0,0.074385,0.114490,0.000000,0.000000,0.017857,0.109375,1.000000e+00
avg_payment_ratio,339575.0,1.360331,28.429260,0.333333,0.955224,1.000000,1.000000,8.482446e+03
total_payment_amount,339587.0,690494.226221,930897.686866,0.000000,133200.742500,324803.520000,849730.905000,3.268928e+07


> Ortalama müşterinin taksitlerinin yaklaşık **%7**'si gecikmiş.

In [8]:
installment_features[
    "avg_payment_ratio"
].describe()

count    339575.000000
mean          1.360331
std          28.429260
min           0.333333
25%           0.955224
50%           1.000000
75%           1.000000
max        8482.446155
Name: avg_payment_ratio, dtype: float64

+ 50% = 1.00

+ 75% = 1.00


> Müşterilerin büyük bölümü tam ödemesini yapmış.

In [9]:
import numpy as np

df = installments.copy()

df["payment_ratio"] = (
    df["AMT_PAYMENT"]
    /
    df["AMT_INSTALMENT"].replace(
        0,
        np.nan
    )
)

np.isinf(
    df["payment_ratio"]
).sum()

0

In [10]:
import numpy as np

np.isinf(
    installment_features["avg_payment_ratio"]
).sum()

0

In [11]:
import pandas as pd

pos = pd.read_csv(
    "../data/raw/POS_CASH_balance.csv"
)

print(pos.shape)

pos.head()

(10001358, 8)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [12]:
pos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   SK_ID_PREV             int64  
 1   SK_ID_CURR             int64  
 2   MONTHS_BALANCE         int64  
 3   CNT_INSTALMENT         float64
 4   CNT_INSTALMENT_FUTURE  float64
 5   NAME_CONTRACT_STATUS   object 
 6   SK_DPD                 int64  
 7   SK_DPD_DEF             int64  
dtypes: float64(2), int64(5), object(1)
memory usage: 610.4+ MB


In [13]:
pos.describe().T

,count,mean,std,min,25%,50%,75%,max
SK_ID_PREV,10001358.0,1.903217e+06,535846.530722,1000001.0,1434405.0,1896565.0,2368963.0,2843499.0
SK_ID_CURR,10001358.0,2.784039e+05,102763.745090,100001.0,189550.0,278654.0,367429.0,456255.0
MONTHS_BALANCE,10001358.0,-3.501259e+01,26.066570,-96.0,-54.0,-28.0,-13.0,-1.0
CNT_INSTALMENT,9975287.0,1.708965e+01,11.995056,1.0,10.0,12.0,24.0,92.0
CNT_INSTALMENT_FUTURE,9975271.0,1.048384e+01,11.109058,0.0,3.0,7.0,14.0,85.0
SK_DPD,10001358.0,1.160693e+01,132.714043,0.0,0.0,0.0,0.0,4231.0
SK_DPD_DEF,10001358.0,6.544684e-01,32.762491,0.0,0.0,0.0,0.0,3595.0


In [14]:
missing_df = (
    pos.isnull()
       .mean()
       .sort_values(ascending=False)
       .to_frame("missing_pct")
)

missing_df.head(20)

,missing_pct
CNT_INSTALMENT_FUTURE,0.002608
CNT_INSTALMENT,0.002607
SK_ID_PREV,0.000000
SK_ID_CURR,0.000000
MONTHS_BALANCE,0.000000
NAME_CONTRACT_STATUS,0.000000
SK_DPD,0.000000
SK_DPD_DEF,0.000000


In [15]:
pos.columns.tolist()

['SK_ID_PREV',
 'SK_ID_CURR',
 'MONTHS_BALANCE',
 'CNT_INSTALMENT',
 'CNT_INSTALMENT_FUTURE',
 'NAME_CONTRACT_STATUS',
 'SK_DPD',
 'SK_DPD_DEF']

In [2]:
import pandas as pd

credit_card = pd.read_csv(
    "../data/raw/credit_card_balance.csv"
)

print(credit_card.shape)

credit_card.head()


(3840312, 23)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [3]:
credit_card.info()

credit_card.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   SK_ID_PREV                  int64  
 1   SK_ID_CURR                  int64  
 2   MONTHS_BALANCE              int64  
 3   AMT_BALANCE                 float64
 4   AMT_CREDIT_LIMIT_ACTUAL     int64  
 5   AMT_DRAWINGS_ATM_CURRENT    float64
 6   AMT_DRAWINGS_CURRENT        float64
 7   AMT_DRAWINGS_OTHER_CURRENT  float64
 8   AMT_DRAWINGS_POS_CURRENT    float64
 9   AMT_INST_MIN_REGULARITY     float64
 10  AMT_PAYMENT_CURRENT         float64
 11  AMT_PAYMENT_TOTAL_CURRENT   float64
 12  AMT_RECEIVABLE_PRINCIPAL    float64
 13  AMT_RECIVABLE               float64
 14  AMT_TOTAL_RECEIVABLE        float64
 15  CNT_DRAWINGS_ATM_CURRENT    float64
 16  CNT_DRAWINGS_CURRENT        int64  
 17  CNT_DRAWINGS_OTHER_CURRENT  float64
 18  CNT_DRAWINGS_POS_CURRENT    float64
 19  CNT_INSTALMENT_MATURE

,count,mean,std,min,25%,50%,75%,max
SK_ID_PREV,3840312.0,1.904504e+06,536469.470563,1000018.000,1434385.00,1897122.0,2.369328e+06,2843496.000
SK_ID_CURR,3840312.0,2.783242e+05,102704.475133,100006.000,189517.00,278396.0,3.675800e+05,456250.000
MONTHS_BALANCE,3840312.0,-3.452192e+01,26.667751,-96.000,-55.00,-28.0,-1.100000e+01,-1.000
AMT_BALANCE,3840312.0,5.830016e+04,106307.031024,-420250.185,0.00,0.0,8.904669e+04,1505902.185
AMT_CREDIT_LIMIT_ACTUAL,3840312.0,1.538080e+05,165145.699525,0.000,45000.00,112500.0,1.800000e+05,1350000.000
AMT_DRAWINGS_ATM_CURRENT,3090496.0,5.961325e+03,28225.688578,-6827.310,0.00,0.0,0.000000e+00,2115000.000
AMT_DRAWINGS_CURRENT,3840312.0,7.433388e+03,33846.077333,-6211.620,0.00,0.0,0.000000e+00,2287098.315
AMT_DRAWINGS_OTHER_CURRENT,3090496.0,2.881696e+02,8201.989345,0.000,0.00,0.0,0.000000e+00,1529847.000
AMT_DRAWINGS_POS_CURRENT,3090496.0,2.968805e+03,20796.887047,0.000,0.00,0.0,0.000000e+00,2239274.160
AMT_INST_MIN_REGULARITY,3535076.0,3.540204e+03,5600.154122,0.000,0.00,0.0,6.633911e+03,202882.005


In [4]:
missing_df = (
    credit_card.isnull()
               .mean()
               .sort_values(ascending=False)
               .to_frame("missing_pct")
)

missing_df.head(20)

,missing_pct
AMT_PAYMENT_CURRENT,0.199981
AMT_DRAWINGS_ATM_CURRENT,0.195249
CNT_DRAWINGS_POS_CURRENT,0.195249
AMT_DRAWINGS_OTHER_CURRENT,0.195249
AMT_DRAWINGS_POS_CURRENT,0.195249
CNT_DRAWINGS_OTHER_CURRENT,0.195249
CNT_DRAWINGS_ATM_CURRENT,0.195249
CNT_INSTALMENT_MATURE_CUM,0.079482
AMT_INST_MIN_REGULARITY,0.079482
SK_ID_PREV,0.000000


In [3]:
import pandas as pd

bureau_balance = pd.read_csv(
    "../data/raw/bureau_balance.csv"
)

print(bureau_balance.shape)

bureau_balance.head()

(27299925, 3)


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


**Bureau Balance'ın Mantığı**

**b**ureau.csv** → her kredi kaydı

        +

**bureau_balance.csv** → o kredi kaydının aylık geçmişi

ilişkisinden oluşuyor.

>C = Closed


In [4]:
bureau_balance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   SK_ID_BUREAU    int64 
 1   MONTHS_BALANCE  int64 
 2   STATUS          object
dtypes: int64(2), object(1)
memory usage: 624.8+ MB


In [5]:
bureau_balance.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SK_ID_BUREAU,27299925.0,NaN,NaN,NaN,6036297.332974,492348.856904,5001709.0,5730933.0,6070821.0,6431951.0,6842888.0
MONTHS_BALANCE,27299925.0,NaN,NaN,NaN,-30.741687,23.864509,-96.0,-46.0,-25.0,-11.0,0.0
STATUS,27299925,8,C,13646993,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
missing_df = (
    bureau_balance.isnull()
                  .mean()
                  .sort_values(ascending=False)
                  .to_frame("missing_pct")
)

missing_df

,missing_pct
SK_ID_BUREAU,0.0
MONTHS_BALANCE,0.0
STATUS,0.0


In [9]:
bureau_balance.columns.tolist()

['SK_ID_BUREAU', 'MONTHS_BALANCE', 'STATUS']

In [8]:
bureau_balance["STATUS"].value_counts()

STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

| STATUS | Anlam              |
| ------ | ------------------ |
| C      | Closed             |
| X      | No loan activity (Kredi faaliyeti yoK)  |
| 0      | Gecikme yok        |
| 1      | 1-30 gün gecikme   |
| 2      | 31-60 gün gecikme  |
| 3      | 61-90 gün gecikme  |
| 4      | 91-120 gün gecikme |
| 5      | 120+ gün gecikme   |


In [10]:
(
    bureau_balance["STATUS"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

STATUS
C    49.99
0    27.47
X    21.28
1     0.89
5     0.23
2     0.09
3     0.03
4     0.02
Name: proportion, dtype: float64

+ Verinin yaklaşık %77'si (C + 0) normal davranış.
+ Yaklaşık %21'i X (aktivite yok).
+ Gecikmeler çok az ama kredi riski açısından çok değerli.
+ Özellikle 5 (120+ gün gecikme) ciddi sinyal.

In [11]:
bureau_balance["STATUS_NUM"] = (
    bureau_balance["STATUS"]
    .replace(
        {
            "X": 0,
            "C": 0,
            "0": 0,
            "1": 1,
            "2": 2,
            "3": 3,
            "4": 4,
            "5": 5
        }
    )
)

bureau_balance["STATUS_NUM"].describe()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20728\4123441182.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(


count    2.729992e+07
mean     2.385996e-02
std      2.743293e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.000000e+00
Name: STATUS_NUM, dtype: float64

In [12]:
bureau_balance["STATUS_NUM"].value_counts()

STATUS_NUM
0    26956982
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

**Toplam kayıt:** 27,299,925

**Gecikmeli kayıtlar (STATUS_NUM > 0):** 

242347 + 62406 + 23419 + 8924 + 5847 = 342943

**Oran:** 342943 / 27299925 ≈ 0.01256 → %1.26

**Oranı:** 26956982 / 27299925 ≈ 0.98744 →  %98.74

**STATUS_NUM > 0**  → %1.26

**STATUS_NUM = 0**  → %98.74